File that generates covariance matric behind in time. 
For each element in matrix, at time t, returns in the period [t-L, t-1] are used, where L is the lookback period. 
For instance L = 21, meaning the covariance the previous 21 days

In [2]:
import os
import numpy as np
import pandas as pd


def generate_backward_cov_proxy(
    H,
    in_path="../data/log_returns.csv",
    out_dir="../proxy",
    prefix="realized_cov",
    debug=True
):
    """
    Build a backward-looking covariance proxy.

    For each row indexed by date t = dates[k], the stored covariance matrix is computed from:
        returns[k-H : k]
    i.e. returns from t-H to t-1 in positional indexing.

    This means the covariance at date t only uses information available strictly before t.

    Output columns:
        Date, cov_A__A, cov_A__B, ..., cov_Z__Z
    with the full NxN matrix flattened row-wise.
    """

    # Load returns
    df = pd.read_csv(in_path, parse_dates=["Date"]).set_index("Date").sort_index()
    assets = df.columns.tolist()

    R = df.to_numpy()
    dates = df.index
    T, N = R.shape

    # Full matrix column names
    colnames = [
        f"cov_{assets[i]}__{assets[j]}"
        for i in range(N)
        for j in range(N)
    ]

    rows = []
    out_dates = []

    # Compute backward-looking proxy
    # Row at dates[k] uses returns from dates[k-H] ... dates[k-1]
    for k in range(H, T):
        window = R[k - H:k]
        Sigma = (window.T @ window) / H

        # Enforce symmetry
        Sigma = 0.5 * (Sigma + Sigma.T)

        rows.append(Sigma.flatten())
        out_dates.append(dates[k])

    proxy_df = pd.DataFrame(rows, index=out_dates, columns=colnames)
    proxy_df.index.name = "Date"

    # Save
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{prefix}_l{H}.csv")
    proxy_df.to_csv(out_path)

    print("Saved:", out_path)
    print("Proxy shape:", proxy_df.shape)
    print("Proxy date range:", proxy_df.index.min().date(), "->", proxy_df.index.max().date())

    # Optional sanity check
    if debug:
        k = H
        print("\nSanity check:")
        print("row date t:", dates[k].date())
        print("window start:", dates[k - H].date())
        print("window end:", dates[k - 1].date())

        manual = (R[k - H:k].T @ R[k - H:k]) / H
        stored = proxy_df.iloc[0].to_numpy().reshape(N, N)

        diff = np.abs(manual - stored).max()
        print("max difference manual vs stored:", diff)

    return proxy_df


backward_proxy_21 = generate_backward_cov_proxy(
    H=21,
    in_path="../data/log_returns.csv",
    out_dir="../proxy",
    prefix="realized_cov",
    debug=True
)

backward_proxy_21.head()

Saved: ../proxy/realized_cov_l21.csv
Proxy shape: (2183, 64)
Proxy date range: 2017-04-27 -> 2025-12-31

Sanity check:
row date t: 2017-04-27
window start: 2017-03-28
window end: 2017-04-26
max difference manual vs stored: 0.0


,cov_US_EQUITY__US_EQUITY,cov_US_EQUITY__INTL_EQUITY,cov_US_EQUITY__US_BONDS,cov_US_EQUITY__INTL_BONDS,cov_US_EQUITY__US_REIT,cov_US_EQUITY__INTL_REIT,cov_US_EQUITY__GOLD,cov_US_EQUITY__BTC,cov_INTL_EQUITY__US_EQUITY,cov_INTL_EQUITY__INTL_EQUITY,...,cov_GOLD__GOLD,cov_GOLD__BTC,cov_BTC__US_EQUITY,cov_BTC__INTL_EQUITY,cov_BTC__US_BONDS,cov_BTC__INTL_BONDS,cov_BTC__US_REIT,cov_BTC__INTL_REIT,cov_BTC__GOLD,cov_BTC__BTC
Date,,,,,,,,,,,,,,,,,,,,,
2017-04-27,0.000021,0.000016,-0.000005,-0.000002,0.000006,0.000008,-0.000015,0.000032,0.000016,0.000033,...,0.000034,0.000007,0.000032,0.000003,0.000013,0.000004,0.000040,0.000044,0.000007,0.000559
2017-04-28,0.000019,0.000016,-0.000004,-0.000002,0.000004,0.000008,-0.000014,0.000033,0.000016,0.000033,...,0.000034,0.000002,0.000033,0.000003,0.000014,0.000009,0.000041,0.000046,0.000002,0.000597
2017-05-01,0.000019,0.000015,-0.000005,-0.000002,0.000005,0.000009,-0.000015,0.000033,0.000015,0.000033,...,0.000034,0.000003,0.000033,0.000005,0.000015,0.000010,0.000043,0.000047,0.000003,0.000595
2017-05-02,0.000019,0.000016,-0.000005,-0.000002,0.000005,0.000010,-0.000015,0.000045,0.000016,0.000034,...,0.000035,-0.000038,0.000045,0.000018,0.000004,0.000005,0.000067,0.000052,-0.000038,0.000868
2017-05-03,0.000019,0.000016,-0.000005,-0.000002,0.000006,0.000010,-0.000014,0.000049,0.000016,0.000034,...,0.000035,-0.000044,0.000049,0.000027,0.000004,0.000008,0.000054,0.000056,-0.000044,0.000801
